In [ ]:
%load_ext autoreload
%autoreload 2

# Fetch News from Hacker News

This notebook demonstrates a simple example of using Neo's agentic workflow with the Playwright MCP server to fetch today's top news from Hacker News.

This is a much simpler example compared to the movie_tickets example - no cookies, no anti-bot strategies, just basic web navigation and content extraction.

## Setup MCP Client with Playwright

Initialize the Playwright MCP client. Unlike the movie_tickets example, we don't need cookies or special configuration.

In [ ]:
from neo.mcp.client import MCPClient

# Initialize Playwright MCP client (no cookies needed!)
playwright_client = MCPClient(
    name="playwright",
    command="npx",
    args=["-y", "@playwright/mcp@latest"]
)

# Connect to the server
await playwright_client.aconnect()

print("✅ Connected to Playwright MCP server")
print(f"Available tools: {len(playwright_client.tools)}")

## Use Neo to Fetch News

Create a Neo task that will navigate to Hacker News and extract the top 10 news items.

In [ ]:
from neo.agentic.neo import Neo
from neo.agentic.task import ModelTask
from neo.agentic.instruction import Instruction, ModelConfigs, OtherConfigs

# Create a task to fetch news from Hacker News
fetch_news_task = ModelTask(
    id="fetch_hacker_news",
    user_input="""Navigate to https://news.ycombinator.com and extract the top 10 news items.
    
For each news item, get:
- Title
- Points (score)
- Author
- Link URL

Present the results in natural language, describing what the top stories are about today.""",
    instruction=Instruction(
        model_configs=ModelConfigs(
            model="gpt-5.2",
        ),
        content="""You are a web browsing assistant that extracts information from websites.
        
Use the Playwright tools to:
1. Navigate to the URL
2. Use browser_snapshot to capture the page content
3. Extract the requested information from the snapshot
4. Format the results in a clear, natural language summary

Keep it simple - no need for complex strategies.""",
        other_configs=OtherConfigs(
            mcp_clients=[playwright_client]
        )
    ),
)

# Create Neo instance and run the task
neo = Neo(
    tasks=fetch_news_task,
    max_tool_execution_rounds=10
)

# Execute the task
result_thread = await neo.run_all()

# Display results
result_thread.display()

## View Task Status

Check the completion status of the task.

In [ ]:
neo.display_task_status()

## Cleanup

Close the MCP client connection.

In [ ]:
# Close MCP client connection
await playwright_client.aclose()
print("Disconnected from Playwright MCP server")

## Notes

**How it works:**
- Neo manages task execution with the AI model
- The model uses Playwright MCP tools to navigate and extract content
- No cookies or anti-bot strategies needed for Hacker News
- Results are formatted in natural language as requested

**Comparison with movie_tickets example:**
- **fetch_news**: Simple demonstration - direct navigation, basic content extraction
- **movie_tickets**: Complex real-world example - cookies, anti-bot strategies, Cloudflare bypass

**When to use which example:**
- Start with fetch_news to understand Neo + MCP basics
- Graduate to movie_tickets when you need to handle bot protection